## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [2]:
load_dotenv(override=True)

True

In [3]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

### It's easy to use any models with OpenAI compatible endpoints

In [4]:

# ----------------------- Olama -----------------------
OLLAMA_BASE_URL = "http://localhost:11434/v1"
LAMMA_DEEP_MODEL = "deepseek-r1:1.5b"   #  1.1 GB
LAMMA_1B_MODEL="llama3.2:1b"            #  1.3 GB
LAMA_QWEN_MODEL = "qwen2.5:3b"          #  1.9 GB
LAMMA_MODEL="llama3.2"                  #  2.0 GB
LAMA_PHI_MODEL = "phi3:latest"          #  2.2 GB
LAMMA_GEMMA = "gemma3:4b"               #  3.3 GB
LAMA_3_1_MODEL = "llama3.1:8b"          #  4.9 GB
LAMMA_GPT_MODEL = "gpt-oss:20b"         # 13.0 GB
LAMMA_GEMMA_4="gemma4:31b-cloud"
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ok_to_pass")
ollama_model = OpenAIChatCompletionsModel(model=LAMMA_GEMMA_4, openai_client=ollama_client)

# ----------------------- Gemini -----------------------
google_api_key = os.getenv('GOOGLE_API_KEY')
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_LITE_MODEL = "gemini-2.5-flash-lite"
GEMINI_MODEL = "gemini-2.5-flash"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model=GEMINI_MODEL, openai_client=gemini_client)

# ----------------------- GROQ -----------------------
groq_api_key = os.getenv('GROQ_API_KEY')
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_LAMA_MODEL = "llama-3.3-70b-versatile"
GROQ_GPT_120_MODEL = "openai/gpt-oss-120b"
GROQ_GPT_20_MODEL = "openai/gpt-oss-20b"
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
groq_model = OpenAIChatCompletionsModel(model=GROQ_GPT_20_MODEL, openai_client=groq_client)

# ----------------------- Z.AI -----------------------
glm_api_key = os.getenv('GLM_Z_API_KEY')
GLM_BASE_URL = "https://api.z.ai/api/paas/v4/"
GLM_7 = "glm-4.7-flash"
GLM_5 = "glm-4.5-flash"
glm_client = AsyncOpenAI(base_url=GLM_BASE_URL, api_key=glm_api_key)
glm_model = OpenAIChatCompletionsModel(model=GLM_7, openai_client=glm_client)

# ----------------------- Openrouter -----------------------
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_GLM_MODEL = "z-ai/glm-4.5"
OPENROUTER_GEMINI_MODEL = "google/gemini-2.5-flash-image"  	
OPENROUTER_GEMINI_LITE_MODEL = "google/gemini-2.5-flash-lite"
OPENROUTER_GEMMA_3_MODEL = "google/gemma-3n-e2b-it:free"
OPENROUTER_GEMMA_3_27_MODEL = "google/gemma-3-27b-it:free"
OPENROUTER_GEMMA_4_26_MODEL = "google/gemma-4-26b-a4b-it:free"
OPENROUTER_GEMMA_4_MODEL = "google/gemma-4-31b-it:free"
OPENROUTER_GPT_4_MINI_MODEL = "openai/gpt-4o-mini"
OPENROUTER_GPT_120_MODEL = "openai/gpt-oss-120b"
OPENROUTER_LLAMA_3_3_MODEL = "meta-llama/llama-3.3-70b-instruct"
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
openrouter_model = OpenAIChatCompletionsModel(model=OPENROUTER_GPT_4_MINI_MODEL, openai_client=openrouter_client)

In [5]:
sales_agent1 = Agent(name="Groq Sales Agent", instructions=instructions1, model=groq_model)
sales_agent2 =  Agent(name="Gemini Sales Agent", instructions=instructions2, model=gemini_model)
sales_agent3  = Agent(name="Ollama Sales Agent",instructions=instructions3,model=ollama_model)

In [6]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [7]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("marcelo.marques@vitarts.com.br")  # Change to your verified sender
    to_email = To("mgmarques3000@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [8]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=openrouter_model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=openrouter_model)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [9]:
email_tools = [subject_tool, html_tool, send_html_email]

In [10]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model=glm_model,
    handoff_description="Convert an email to HTML and send it")

In [11]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [12]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try until up yuo handoff.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

groq_lama_model = OpenAIChatCompletionsModel(model=GROQ_LAMA_MODEL, openai_client=groq_client)


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=groq_lama_model)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

## Check out the trace:

https://platform.openai.com/traces

In [13]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model=openrouter_model
)

In [14]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [15]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model=ollama_model,
    input_guardrails=[guardrail_against_name]
    )

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

## Check out the trace:

https://platform.openai.com/traces

In [16]:

message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>